In [1]:
import requests
import time
import json
import os

In [2]:
QUERY = """
query ($page: Int, $type: MediaType) {
  Page (page: $page, perPage: 50) {
    pageInfo {
      hasNextPage
      currentPage
    }
    media (type: $type, sort: ID) {
      id
      idMal
      title {
        romaji
        english
      }
      synonyms
      stats {
        scoreDistribution {
          score
          amount
        }
      }
      type
      countryOfOrigin
      format
      status
      startDate { year month day }
      endDate { year month day }
      season
      seasonYear
      episodes
      chapters
      volumes
      duration
      averageScore
      popularity
      trending
      description(asHtml: false)
      genres
      studios {
        nodes {
          name
        }
      }
      tags {
        name
        rank
      }
      isAdult
      source
      coverImage {
        large
        color
      }
      bannerImage
      trailer {
        id
        site
      }
      siteUrl
    }
  }
}
"""

URL = 'https://graphql.anilist.co'

def fetch_and_save_media(media_type, filename):
    """
    Fetches data and appends it line-by-line to a JSONL file to prevent data loss.
    """
    page = 1
    has_next_page = True
    total_saved = 0

    print(f"\n--- Starting extraction for {media_type} ---")
    print(f"Data will be appended to {filename} in real-time.")

    # Open file in append mode
    with open(filename, 'a', encoding='utf-8') as f:
        while has_next_page:
            variables = {
                'page': page,
                'type': media_type
            }

            try:
                response = requests.post(URL, json={'query': QUERY, 'variables': variables})
                
                # Strict Rate Limit Handling
                if 'x-ratelimit-remaining' in response.headers:
                    remaining = int(response.headers['x-ratelimit-remaining'])
                    if remaining < 15:
                        print(f"[Rate Limit] Only {remaining} requests left. Sleeping for 15 seconds...")
                        time.sleep(15)

                # Handle HTTP errors
                if response.status_code != 200:
                    print(f"Error {response.status_code}: {response.text}")
                    print("Sleeping for 60 seconds before retrying page...")
                    time.sleep(60)
                    continue 

                data = response.json()
                
                page_info = data['data']['Page']['pageInfo']
                media_list = data['data']['Page']['media']
                
                # Save each item as a JSON string on a new line
                for item in media_list:
                    f.write(json.dumps(item, ensure_ascii=False) + '\n')
                    total_saved += 1
                
                print(f"[{media_type}] Page {page} saved. Total items: {total_saved}")
                
                has_next_page = page_info['hasNextPage']
                page += 1
                
                # Standard throttle to play nice with their servers
                time.sleep(0.5) 

            except Exception as e:
                print(f"CRITICAL ERROR on page {page}: {e}")
                print("Script stopped, but all previous data is safely saved to disk.")
                break

    print(f"Finished {media_type} extraction. Total saved: {total_saved}")

if __name__ == "__main__":
    # Create distinct files for Anime and Manga
    anime_file = "anilist_anime_complete.jsonl"
    manga_file = "anilist_manga_complete.jsonl"
    
    # Run the extractions
    fetch_and_save_media("ANIME", anime_file)
    fetch_and_save_media("MANGA", manga_file)
    
    print("\nAll operations complete.")


--- Starting extraction for ANIME ---
Data will be appended to anilist_anime_complete.jsonl in real-time.
[ANIME] Page 1 saved. Total items: 50
[ANIME] Page 2 saved. Total items: 100
[ANIME] Page 3 saved. Total items: 150
[ANIME] Page 4 saved. Total items: 200
[ANIME] Page 5 saved. Total items: 250
[ANIME] Page 6 saved. Total items: 300
[ANIME] Page 7 saved. Total items: 350
[ANIME] Page 8 saved. Total items: 400
[ANIME] Page 9 saved. Total items: 450
[ANIME] Page 10 saved. Total items: 500
[ANIME] Page 11 saved. Total items: 550
[ANIME] Page 12 saved. Total items: 600
[ANIME] Page 13 saved. Total items: 650
[ANIME] Page 14 saved. Total items: 700
[ANIME] Page 15 saved. Total items: 750
[Rate Limit] Only 14 requests left. Sleeping for 15 seconds...
[ANIME] Page 16 saved. Total items: 800
[ANIME] Page 17 saved. Total items: 850
[ANIME] Page 18 saved. Total items: 900
[ANIME] Page 19 saved. Total items: 950
[ANIME] Page 20 saved. Total items: 1000
[ANIME] Page 21 saved. Total items: 105